<a href="https://colab.research.google.com/github/Szymoniakfoltynson/ai-course-friday/blob/main/Klasyfikator_opinii_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikator opinii oparty na modelu BERT
1. Konfiguracja środowiska
2. Definicja kategorii i ustalenie ich reprezentacji wektorowych
3. Ręczna klasyfikacja + interface gradio
4. Klasyfikacja i analiza zbioru danych
5. Wykresy i prezentacja zbioru danych
6. Import zbioru danych wejściowych i generacja wyników



### 1. Konfiguracja środowiska

In [1]:
!pip install sentence-transformers matplotlib

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 2. Definicja kategorii i ustalenie ich reprezentacji wektorowych

In [24]:
categories = {
    "pozytywna": [
        "To był świetny produkt.",
        "Bardzo polecam, wszystko super.",
        "Jestem zadowolony.",
        "Naprawdę dobra jakość.",
        "Gra była bardzo przyjemna.",
        "Rewelacyjna rozgrywka i grafika.",
        "Bardzo satysfakcjonująca mechanika gry.",
        "Czułem dużo frajdy podczas grania.",
        "Produkt spełnił wszystkie moje oczekiwania.",
        "Świetna obsługa i szybka dostawa.",
        "Wszystko działa bez problemu.",
        "Jakość wykonania jest bardzo wysoka.",
        "To jedna z najlepszych rzeczy, jakie kupiłem.",
        "Bardzo udany zakup.",
        "Gra wciągnęła mnie na wiele godzin.",
        "Muzyka i klimat były fantastyczne.",
        "Postacie były bardzo ciekawe i dobrze napisane.",
        "Sterowanie było intuicyjne i wygodne.",
        "Zdecydowanie kupiłbym to jeszcze raz.",
        "Produkt był wart swojej ceny."
    ],

    "negatywna": [
        "Nie polecam, bardzo słaba jakość.",
        "To była strata pieniędzy.",
        "Jestem niezadowolony.",
        "Znudziła mi się po jednym dniu.",
        "Gra była nudna i frustrująca.",
        "Rozgrywka była monotonna i męcząca.",
        "Sterowanie działało bardzo źle.",
        "Zawiodłem się całkowicie.",
        "Produkt szybko się zepsuł.",
        "Jakość wykonania jest bardzo słaba.",
        "Obsługa klienta była fatalna.",
        "Gra miała mnóstwo błędów.",
        "Nie działało tak, jak powinno.",
        "To najgorszy zakup od dawna.",
        "Czułem tylko frustrację podczas używania.",
        "Grafika wyglądała bardzo słabo.",
        "Gra często się zawieszała.",
        "Produkt nie był wart swojej ceny.",
        "Mechanika była irytująca i nieprzemyślana.",
        "Nie kupiłbym tego ponownie."
    ],

    "mieszana": [
        "Produkt jest niezły, ale ma kilka wad.",
        "Gra była ciekawa, chociaż momentami nudna.",
        "Jakość jest dobra, ale cena za wysoka.",
        "Podobała mi się grafika, ale sterowanie było słabe.",
        "Obsługa była miła, ale dostawa trwała za długo.",
        "Nie wszystko działało idealnie, ale ogólnie jestem zadowolony.",
        "Gra miała świetny klimat, ale była za krótka.",
        "Produkt wygląda dobrze, ale mógłby być solidniejszy.",
        "Mechanika była ciekawa, ale szybko się powtarzała.",
        "Film był interesujący, choć zakończenie mnie rozczarowało.",
        "Sprzęt działa poprawnie, ale spodziewałem się czegoś więcej.",
        "Były zarówno dobre, jak i słabe momenty.",
        "Gra miała dobre pomysły, ale wykonanie nie zawsze było udane.",
        "Nie jestem zachwycony, ale też nie żałuję zakupu.",
        "Produkt ma swoje plusy i minusy.",
        "Muzyka była świetna, ale fabuła bardzo przeciętna.",
        "To nie był zły zakup, ale nie był też wyjątkowy.",
        "Część funkcji działa bardzo dobrze, a część bardzo słabo.",
        "Było kilka irytujących problemów, ale dało się z tego korzystać.",
        "Ogólnie jest okej, choć wymaga dopracowania."
    ]
}
# =================================


# MIEJSCE NA KOD
category_vectors = {} # Initialize an empty dictionary to store category embeddings

for category_name, examples_list in categories.items(): # Unpack into two variables
    embeddings = model.encode(examples_list)
    category_vectors[category_name] = np.mean(embeddings, axis=0)

### 3. Ocena ręcznie wpisanej opinii + gradio

In [26]:
# =================================
# miejsce na funkcję classify_opinion
def classify_opinion(text):
    embedding = model.encode([text])[0]
    similarities = {
        category: cosine_similarity(
            [embedding], [vector]
            )[0][0] for category, vector in category_vectors.items()
        }
    predicted_category = max(similarities, key=similarities.get)
    response = f"Opinia: {text} - {predicted_category}"
    # return predicted_category
    return similarities


In [28]:
# =================================
# miejsce na gradio
gr.Interface(
    fn=classify_opinion,
    inputs=gr.Textbox(label="Wpisz opinię"),
    outputs=gr.Label(label="Wynik klasyfikacji"),
    title="Klasyfikator opinii",
    description="Podaj opinię i wybierz klasyfikację"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d4dbd7a2c4b30da6e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 4. Klasyfikacja i analiza zbioru danych

In [ ]:
def load_opinions(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def classify_opinion_vector(opinion):
    embedding = model.encode([opinion])[0]
    sims = {cat: cosine_similarity([embedding], [vec])[0][0] for cat, vec in category_vectors.items()}
    predicted = max(sims, key=sims.get)
    return predicted, sims

def analyze_opinions(file_path):
    opinions = load_opinions(file_path)
    results = {'pozytywna': 0, 'negatywna': 0, 'mieszana': 0}
    all_info = []
    for opinion in opinions:
        cat, sims = classify_opinion_vector(opinion)
        results[cat] += 1
        all_info.append((opinion, cat, sims))
    return results, all_info

### 4. Wykresy i prezentacja zbioru danych

In [ ]:
def show_results(results):
    labels = list(results.keys())
    counts = list(results.values())
    total = sum(counts)
    percentages = [100 * c / total for c in counts]
    plt.bar(labels, percentages, color=["green", "red"])
    plt.title("Procent opinii w każdej kategorii")
    plt.ylabel("Procent")
    plt.ylim(0, 100)
    plt.show()

def show_detailed_results(details):
    for i, (opinia, kategoria, podobienstwa) in enumerate(details):
        print(f"\n  Opinia {i+1}: {opinia}")
        print(f" Klasyfikacja: {kategoria.upper()}")
        for kat, val in podobienstwa.items():
            print(f"  → {kat}: {val:.4f}")

### 6. Import zbioru danych wejściowych i generacja wyników

In [ ]:
file_path = "opinie.txt"
results, details = analyze_opinions(file_path)
show_results(results)
show_detailed_results(details)

### 7. Interface do testowania z plikiem (BONUS)

In [ ]:
def classify_file_interface(file):
    opinions = load_opinions(file.name)
    results, all_info = analyze_opinions(file.name)

    total = results['pozytywna'] + results['negatywna']
    summary = (
        f" Liczba opinii: {total}\n"
        f" Pozytywne: {results['pozytywna']} ({results['pozytywna'] / total:.1%})\n"
        f" Negatywne: {results['negatywna']} ({results['negatywna'] / total:.1%})\n\n"
    )

    details_text = ""
    for i, (opinia, kategoria, podobienstwa) in enumerate(all_info):
        details_text += f" Opinia {i+1}: {opinia}\n"
        details_text += f" Klasyfikacja: {kategoria.upper()}\n"
        for kat, val in podobienstwa.items():
            details_text += f"→ {kat}: {val:.4f}\n"
        details_text += "\n"

    return summary + details_text

In [ ]:
gr.Interface(
    fn=classify_file_interface,
    inputs=gr.File(label="Wgraj plik z opiniami (.txt)"),
    outputs=gr.Textbox(label="Wyniki klasyfikacji", lines=30),
    title="Klasyfikator opinii z pliku",
    description="Wgraj plik tekstowy (.txt)"
).launch()